# Fase Entrenamiento
## Jabes Esteban Monroy Becerra
### PID - 19 de junio de 2026

# Entrenamiento y exportación — Clasificador de enfermedades en plantas

Notebook para Google Colab (GPU). Descarga PlantVillage, entrena MobileNetV2,
genera las gráficas y exporta el modelo a ExecuTorch (`.pte`).

## 1. Clonar el repositorio

In [ ]:
%cd /content
!rm -rf /content/proyecto
!git clone https://github.com/JabesMonroy/plant-disease-classifier.git /content/proyecto
%cd /content/proyecto

## 2. Instalar dependencias

In [ ]:
!pip install -q -U kaggle

## 3. Descargar el dataset PlantVillage


In [ ]:
import json

with open('kaggle.json', 'w') as f:
    json.dump({'username': 'jabesmonroy',
               'key': 'KGAT_a94fd94447971336d25c0cfa26fe01e8'}, f)

In [ ]:
# Normaliza el token al formato que espera la CLI (campo 'key') y descarga el dataset.
import json, os

cfg = json.load(open('kaggle.json'))
key = cfg.get('key') or cfg.get('access_token')
os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
with open(os.path.expanduser('~/.kaggle/kaggle.json'), 'w') as f:
    json.dump({'username': cfg['username'], 'key': key}, f)
os.chmod(os.path.expanduser('~/.kaggle/kaggle.json'), 0o600)

!kaggle datasets download -d abdallahalidev/plantvillage-dataset -p data --unzip

In [ ]:
# Localiza automáticamente la carpeta 'color' del dataset descargado.
import glob
data_dir = glob.glob('data/**/color', recursive=True)[0]
print('Dataset en:', data_dir)

## 4. Entrenar el modelo
Genera `artifacts/best.pt`, las gráficas y `labels.json`.

In [ ]:
!python src/train.py \
    --data-dir "{data_dir}" \
    --out-dir artifacts \
    --epochs 10 \
    --batch-size 64

## 5. Ver las gráficas de entrenamiento

In [ ]:
from IPython.display import Image, display
for img in ['curva_perdida.png','curva_accuracy.png','matriz_confusion.png']:
    display(Image(filename=f'artifacts/{img}'))

## 6. Exportar a ExecuTorch
Instala ExecuTorch (fija torch/torchvision a las versiones que exige) y crea `space/model.pte`.

In [ ]:
!pip install -q executorch==0.4.0 torch==2.5.0 torchvision==0.20.0

In [ ]:
!python export/export_executorch.py \
    --checkpoint artifacts/best.pt \
    --labels artifacts/labels.json \
    --output space/model.pte

## 7. Descargar resultados
Descarga el modelo y las etiquetas para el Space, y las gráficas para las diapositivas.

In [ ]:
from google.colab import files
for f in ['space/model.pte', 'space/labels.json',
          'artifacts/curva_perdida.png', 'artifacts/curva_accuracy.png',
          'artifacts/matriz_confusion.png']:
    files.download(f)